### SCPT to VS all-in-one tool (Beta)

In [1]:
# Import packages for use:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Times New Roman'
pd.set_option('display.max_columns', None)

# All SCPT functions are in scpt_tool.py at the repository root
sys.path.insert(0, '..')
from scpt_tool import compute_profiles, plot_profiles, compute_VsZ, ZHANG_A


### IMPORT TRAVEL TIME DATA

In [2]:
'''

read travel time data csv as dataframe, 
make sure the last row is not NAN in the .csv file 
and unit of depth is meter and unit of travel time is millisecond.
change the name to the target input file. 

'''

# TT_DATA = pd.read_csv('downhole_KS.csv')
# TT_DATA = pd.read_csv('SCPT_WYE_77_tt.csv')
TT_DATA = pd.read_csv('avn_1_tt.csv')


TT_depth = np.asarray(TT_DATA['depth (m)'])
Meas_TT = np.asarray(TT_DATA['traveltime (ms)'])

### IMPORT CPT DATA

In [3]:
'''
read CPT data csv as dataframe, 
make sure the last row is not NAN in the .csv file 
and unit of depth is meter and unit of qt and fs are MPa.
change the name to the target input file. 

'''

# CPT_DATA = pd.read_csv('SCPT_WYE_77.csv')
CPT_DATA = pd.read_csv('avn_1.csv')

CPT_depth = np.asarray(CPT_DATA['depth (m)'])
CPT_qt = np.asarray(CPT_DATA['qt (MPa)'])
CPT_fs = np.asarray(CPT_DATA['fs (MPa)'])
CPT_qc = np.asarray(CPT_DATA['qc (MPa)'])

valid_mask = ~np.isnan(CPT_depth) & ~np.isnan(CPT_qt) & ~np.isnan(CPT_fs)

CPT_depth = CPT_depth[valid_mask]
CPT_qt = CPT_qt[valid_mask]
CPT_fs = CPT_fs[valid_mask]
CPT_qc = CPT_qc[valid_mask]

# Enter depth of groundwater table in meter, if the groundwater table is not encountered, please enter a value larger than the maximum depth of CPT data.
print (np.max (CPT_depth))
dGWT = 31

30.55


### Calculate Vs with all four methods

**Zhang et al.** (computed with qc): ln($V_S$) = 0.224117 * ln($K_c$ * $Q_{tn}$) + m($I_c$) * ln($\sigma_v'$/$p_a$) + a

$K_c$ = 1 + 5 / (1 + exp(-5.482742 * $I_c$ + 12.320768)), m($I_c$) = 0.1 + 0.4 / (1 + exp(-1.278293 * $I_c$ + 3.385425)), a = geology-specific offset. $Q_{tn}$ and $I_c$ are inverse-filtered (Boulanger and DeJong, 2018).

**Zhang et al. (site specific)**: same model, with "a" fitted to the measured travel time of each layer (each CPT point represents an equal share of its layer).

**Robertson (2012)** (computed with qt): ln($V_S$) = 1.93 + 0.5 * ln($Q_{tn}$) + 0.25 * ln($\sigma_v'$/$p_a$) + 0.63 * $I_c$

**Layer-based regression (LBR)**: piecewise-linear fit of the travel time data, with the number of layers chosen by the cost function.

In [4]:
"""Enter the geology type of the site, the options are the keys of ZHANG_A in scpt_tool.py:
'Beach and Dune', 'Floodplain', 'Glacial', 'Lagoonal', 'Valley Train', 'af/Qi', 'Qal1', 'Qal2', 'Qal3', 'Qoa', 'Qs'"""

geology = "Qal1"

p = compute_profiles(CPT_depth, CPT_qc, CPT_qt, CPT_fs, dGWT, TT_depth, Meas_TT, geology)
print(f"geology = {geology}  |  a_geo = {p['a_geo']:.4f}  |  a_ss = {p['a_site_specific']:.4f}")

VsZ_results, VsZp = compute_VsZ(p)
pd.DataFrame([{'Method': k, 'Z (m)': Z, 'VsZ (m/s)': v} for k, (v, Z) in VsZ_results.items()])

geology = Qal1  |  a_geo = 4.3502  |  a_ss = 4.1438


,Method,Z (m),VsZ (m/s)
0,Zhang et al.,30.550000,303.120330
1,Zhang et al. (ss),30.550000,246.572108
2,Robertson (2012),30.550000,254.642132
3,LBR,30.348739,248.519202


##### Export site specific results to CSV

In [5]:
# Export results to CSV
site_specific_results = pd.DataFrame({
    "depth (m)": p['CPT_depth'],
    "velocity (m/s)": p['site_specific_velocity']
})
site_specific_results.to_csv('vs_ss_results.csv', index=False)

### GUI for all four methods

In [6]:
import ipywidgets as widgets
from ipywidgets import HTML, Layout, HBox, VBox, Button
from IPython.display import display

%matplotlib widget

# Setup
style = {'description_width': 'initial'}

velocity_updated = []
top_depth_updated = []
bottom_depth_updated = []

VsZ_input = widgets.FloatText(value=1.0, description='VsZ Direct Interpretation (LBR) (m/s)', style=style)
VsZ_glob_input = widgets.FloatText(value=1.0, description='VsZ Zhang et al. (m/s)', style=style)
VsZ_ss_input = widgets.FloatText(value=1.0, description='VsZ Zhang et al. (ss) (m/s)', style=style)
VsZ_rob_input = widgets.FloatText(value=1.0, description='VsZ Robertson (2012) (m/s)', style=style)

button_layout = widgets.Layout(width='600px', height='40px')
EXPORT = Button(description='Export Result', layout = button_layout)
MANUAL_REVISE = Button(description='Manual Revision', layout = button_layout)

# Dynamic checkbox container (empty to start)
checkbox_box = widgets.VBox(layout=Layout(flex_flow='row wrap', width='100%'))

box_layout_2 = widgets.Layout(width='100%', border='solid 2px', padding='20px')
SEARCH = VBox([
    HTML('<center><font size="+1.5"><b>CPT-Vs Interpretation Tool</b>'),
    HBox([VsZ_input, VsZ_glob_input, VsZ_ss_input, VsZ_rob_input], layout=Layout(width='100%')),
    HBox([EXPORT, MANUAL_REVISE], layout=Layout(width='100%')),
    HTML('<center><font size="+0.5">Slope Break Locations - Direct Interpretation'),
    checkbox_box  # add the checkboxes below the buttons
], layout=box_layout_2)

out = widgets.Output()

def Run_and_Plot(slope_break_method = 0, breakpoints = None):

    global velocity_updated
    global top_depth_updated
    global bottom_depth_updated

    plt.close('all')
    p = compute_profiles(CPT_depth, CPT_qc, CPT_qt, CPT_fs, dGWT, TT_depth, Meas_TT, geology,
                         slope_break_method = slope_break_method, breakpoints = breakpoints)

    print(f"geology = {geology}  |  a_geo = {p['a_geo']:.4f}  |  a_ss = {p['a_site_specific']:.4f}")
    fig = plot_profiles(p)
    fig.savefig('cpt_vs_interpretation.png', dpi=300, bbox_inches='tight')

    VsZ_results, VsZp = compute_VsZ(p)
    VsZ_input.value = np.round(VsZ_results['LBR'][0], 0)
    VsZ_glob_input.value = np.round(VsZ_results['Zhang et al.'][0], 0)
    VsZ_ss_input.value = np.round(VsZ_results['Zhang et al. (ss)'][0], 0)
    VsZ_rob_input.value = np.round(VsZ_results['Robertson (2012)'][0], 0)

    velocity_updated = p['LBR_velocity']
    top_depth_updated = p['LBR_ztop']
    bottom_depth_updated = p['LBR_zbot']

    return p

# --- Reactive SCPT update ---
def Update_SCPT(_):
    out.clear_output()
    with out:
        p = Run_and_Plot(slope_break_method = 0)

        # Update the checkboxes dynamically
        results = p['results']
        breaks = sorted(set(results.get('candidate_breakpoints', [])))
        used_breaks = [str(round(b, 3)) for b in results.get('breaks', [])]
        checkbox_box.children = [
            widgets.Checkbox(description=str(round(b, 3)), indent=False, value=(str(round(b, 3)) in used_breaks))
            for b in breaks
        ]

def Full_Manual(_):
    out.clear_output()
    with out:
        # Get selected breakpoints
        selected_breakpoints = [float(checkbox.description) for checkbox in checkbox_box.children if checkbox.value]
        if not selected_breakpoints:
            print("No breakpoints selected.")
            return
        # Run full manual regression
        try:
            Run_and_Plot(slope_break_method = 1, breakpoints = selected_breakpoints)
        except ValueError as e:
            print(f"Manual revision failed: {e}")

# Export Function
def Export (_):

    # Save the LBR Vs profile to a CSV file
    try:
        VS_SLOPEBREAK_DF = pd.read_csv('slopebreak_vs.csv')
    except FileNotFoundError:
        VS_SLOPEBREAK_DF = pd.DataFrame(columns=['Velocity', 'Top_depth', 'Bottom_depth'])

    new_data2 = pd.DataFrame({
        'Velocity': velocity_updated,
        'Top_depth': top_depth_updated,
        'Bottom_depth': bottom_depth_updated})
    VS_SLOPEBREAK_DF = pd.concat([VS_SLOPEBREAK_DF, new_data2], ignore_index=True)
    VS_SLOPEBREAK_DF.to_csv('slopebreak_vs.csv', index=False)

# Initial display
display(SEARCH)
display(out)

# Trigger first update manually
Update_SCPT(None)

# Attach triggers to the buttons
EXPORT.on_click(Export)
MANUAL_REVISE.on_click(Full_Manual)

Output()